# Introduction

The goal of this notebook is to showcase how to use and test the pre-trained modified C-GCN model  

The notebook is structured as follows:
1. Loading the model
2. Demonstration of executing the model on the test data
3. Demonstration of executing the model on user provided data


In [1]:
# CD into the parent folder for convinience

%cd ../../src_ra_cgcn
%pwd

/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn


'/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn'

In [2]:
# imports

import random
import argparse

from tqdm import tqdm
import torch

from data.loader import DataLoader, DataLoaderPredict
from model.trainer import GCNTrainer
from utils import torch_utils, scorer, constant, helper
from utils.vocab import Vocab


In [3]:
# loading the model

# set up model loading arguments
args = dict()
args['model_dir'] = f'saved_models/00/'
args['model'] = f'best_model.pt'

args['seed'] = 1234
args['cuda'] = torch.cuda.is_available()
args['cpu'] = False if args['cuda'] else True

# set the seeds
torch.manual_seed(args['seed'])
random.seed(args['seed'])
if args['cuda']:
    torch.cuda.manual_seed(args['seed'])

# load opt
model_file = args['model_dir'] + '/' + args['model']
print("Loading model from {}".format(model_file))
opt = torch_utils.load_config(model_file)
trainer = GCNTrainer(opt)
trainer.load(model_file)

# load vocab
vocab_file = args['model_dir'] + '/vocab.pkl'
vocab = Vocab(vocab_file, load=True)
assert opt['vocab_size'] == vocab.size, "Vocab size must match that in the saved model."


Loading model from saved_models/00//best_model.pt
Finetune all embeddings.


/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn/utils/torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental

Vocab size 50115 loaded from file


/Users/rojsaktumanis/Desktop/Studies/Manchester modules/Text mining/RE CW/CW/uom-relation-extraction/src_ra_cgcn/model/trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feat

In [ ]:
args['data_dir'] = f'dataset/tacred'
args['dataset'] = f'test'

# load data
data_file = opt['data_dir'] + '/{}.json'.format(args['dataset'])
print("Loading data from {} with batch size {}...".format(data_file, opt['batch_size']))
batch = DataLoader(data_file, opt['batch_size'], opt, vocab, evaluation=True)

# helper.print_config(opt)
label2id = constant.LABEL_TO_ID
id2label = dict([(v,k) for k,v in label2id.items()])

predictions = []
all_probs = []
batch_iter = tqdm(batch)
for i, b in enumerate(batch_iter):
    preds, probs, _ = trainer.predict(b)
    predictions += preds
    all_probs += probs

predictions = [id2label[p] for p in predictions]
_, details = scorer.score(batch.gold(), predictions, verbose=False)

# print("Detailed evaluation:")
# print("\n".join([f"{k}: {v}" for k, v in details.items()]))

print("Evaluation ended.")

Loading data from ../data/retacred/test.json with batch size 50...
269 batches created for ../data/retacred/test.json


100%|██████████| 269/269 [00:07<00:00, 37.49it/s]

Precision (micro): 76.230%
   Recall (micro): 72.964%
       F1 (micro): 74.561%
Evaluation ended.


In [10]:
tmp_data = [
    {
        "token": ["He", "has", "served", "as", "a", "policy", "aide", "to", "the", "late", "U.S.", "Senator", "Alan", "Cranston", ",", "as", "National", "Issues", "Director", "for", "the", "2004", "presidential", "campaign", "of", "Congressman", "Dennis", "Kucinich", ",", "as", "a", "co-founder", "of", "Progressive", "Democrats", "of", "America", "and", "as", "a", "member", "of", "the", "international", "policy", "department", "at", "the", "RAND", "Corporation", "think", "tank", "before", "all", "that", "."], 
        "subj_start": 33, 
        "subj_end": 36, 
        "obj_start": 43, 
        "obj_end": 45, 
        "subj_type": "ORGANIZATION", 
        "obj_type": "ORGANIZATION", 
        "stanford_pos": ["PRP", "VBZ", "VBN", "IN", "DT", "NN", "NN", "TO", "DT", "JJ", "NNP", "NNP", "NNP", "NNP", ",", "IN", "NNP", "NNP", "NNP", "IN", "DT", "CD", "JJ", "NN", "IN", "NNP", "NNP", "NNP", ",", "IN", "DT", "NN", "IN", "NNP", "NNPS", "IN", "NNP", "CC", "IN", "DT", "NN", "IN", "DT", "JJ", "NN", "NN", "IN", "DT", "NNP", "NNP", "VB", "NN", "IN", "DT", "DT", "."], 
        "stanford_ner": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "LOCATION", "O", "PERSON", "PERSON", "O", "O", "O", "O", "O", "O", "O", "DATE", "O", "O", "O", "O", "PERSON", "PERSON", "O", "O", "O", "O", "O", "ORGANIZATION", "ORGANIZATION", "ORGANIZATION", "ORGANIZATION", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "ORGANIZATION", "ORGANIZATION", "O", "O", "O", "O", "O", "O"], 
        "stanford_head": [3, 3, 0, 7, 7, 7, 3, 14, 14, 14, 14, 14, 14, 3, 3, 19, 19, 19, 3, 24, 24, 24, 24, 19, 28, 28, 28, 24, 3, 32, 32, 3, 35, 35, 32, 37, 35, 32, 41, 41, 32, 46, 46, 46, 46, 41, 50, 50, 50, 46, 41, 51, 54, 52, 54, 3], 
        "stanford_deprel": ["nsubj", "aux", "ROOT", "case", "det", "compound", "nmod", "case", "det", "amod", "compound", "compound", "compound", "nmod", "punct", "case", "compound", "compound", "nmod", "case", "det", "nummod", "amod", "nmod", "case", "compound", "compound", "nmod", "punct", "case", "det", "nmod", "case", "compound", "nmod", "case", "nmod", "cc", "case", "det", "conj", "case", "det", "amod", "compound", "nmod", "case", "det", "compound", "nmod", "acl", "dobj", "case", "nmod", "dep", "punct"]
    }
]

print(f"Input sentence: {' '.join(tmp_data[0]['token'])}")
print(f"Subject: {' '.join(tmp_data[0]['token'][tmp_data[0]['subj_start']:tmp_data[0]['subj_end']+1])} ({tmp_data[0]['subj_type']})")
print(f"Object: {' '.join(tmp_data[0]['token'][tmp_data[0]['obj_start']:tmp_data[0]['obj_end']+1])} ({tmp_data[0]['obj_type']})")

batch = DataLoaderPredict(tmp_data, opt, vocab)

# helper.print_config(opt)
label2id = constant.LABEL_TO_ID
id2label = dict([(v,k) for k,v in label2id.items()])

predictions = []
all_probs = []
for i, b in enumerate(batch):
    preds, probs, _ = trainer.predict(batch=b, test=True)
    predictions += preds
    all_probs += probs

predictions = [id2label[p] for p in predictions]

print(f"Predicted relation: {predictions[0]}")

Input sentence: He has served as a policy aide to the late U.S. Senator Alan Cranston , as National Issues Director for the 2004 presidential campaign of Congressman Dennis Kucinich , as a co-founder of Progressive Democrats of America and as a member of the international policy department at the RAND Corporation think tank before all that .
Subject: Progressive Democrats of America (ORGANIZATION)
Object: international policy department (ORGANIZATION)
Predicted relation: no_relation
